# 06 — Black-Litterman with LLM Views

Compare five BL variants over the OOS window (2023-01-01 → 2026-04-30):

| Generator | Description |
|---|---|
| `equilibrium` | No views — BL on CAPM equilibrium-implied returns |
| `momentum` | Views = annualised 252-day trailing return per asset |
| `mean_rev` | Views = −1 × long-run 5-year mean (sign-flip = reversion) |
| `llm-anthropic` | LLM views via Anthropic claude-opus-4-7 |
| `llm-openai` | LLM views via OpenAI gpt-5.5 |

All variants share identical BL parameters (τ=0.05, δ=2.5, Ledoit-Wolf cov, monthly refits).

**To run the live backtest:**
```bash
python scripts/run_bl_views_backtest.py --generator equilibrium
python scripts/run_bl_views_backtest.py --generator momentum
python scripts/run_bl_views_backtest.py --generator mean_rev
python scripts/run_bl_views_backtest.py --generator llm-anthropic
python scripts/run_bl_views_backtest.py --generator llm-openai
```

In [ ]:
import sys
sys.path.insert(0, "../src")

import json
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pathlib import Path

from aiam.evaluation.performance import performance_stats

ROOT = Path("..").resolve()
RESULTS_DIR = ROOT / "results" / "llm"

GENERATORS = ["equilibrium", "momentum", "mean_rev", "llm-anthropic", "llm-openai"]
DISPLAY = {
    "equilibrium":  "BL-Equilibrium",
    "momentum":     "BL-Momentum",
    "mean_rev":     "BL-MeanRev",
    "llm-anthropic": "BL-LLM(Anthropic)",
    "llm-openai":   "BL-LLM(OpenAI)",
}
COLORS = {
    "equilibrium":  "#1f77b4",
    "momentum":     "#ff7f0e",
    "mean_rev":     "#2ca02c",
    "llm-anthropic": "#d62728",
    "llm-openai":   "#9467bd",
}

plt.rcParams.update({"figure.dpi": 120, "font.size": 10})
print(f"Results directory: {RESULTS_DIR}")

## 1 — Load results

In [ ]:
returns_dict: dict[str, pd.Series] = {}
diag_dict: dict[str, dict] = {}

for gen in GENERATORS:
    rpath = RESULTS_DIR / f"{gen}_returns.parquet"
    dpath = RESULTS_DIR / f"{gen}_diagnostics.json"
    if rpath.exists():
        df = pd.read_parquet(rpath)
        returns_dict[gen] = df["portfolio_return"]
        if dpath.exists():
            diag_dict[gen] = json.loads(dpath.read_text())
        print(f"  {DISPLAY[gen]:25s} {len(returns_dict[gen]):>5d} OOS days  "
              f"mock={diag_dict.get(gen, {}).get('mock', '?')}")
    else:
        print(f"  [MISSING] {rpath.name}")

print(f"\nLoaded {len(returns_dict)}/{len(GENERATORS)} generators.")

## 2 — Performance comparison table

In [ ]:
rows = []
for gen, ret in returns_dict.items():
    diag = diag_dict.get(gen, {})
    s = performance_stats(ret.dropna())
    rows.append({
        "Strategy":       DISPLAY[gen],
        "Sharpe":         s["sharpe_ratio"],
        "Ann Ret":        s["annualized_return"],
        "Ann Vol":        s["annualized_volatility"],
        "Max DD":         s["max_drawdown"],
        "Turnover/refit": diag.get("mean_turnover_per_refit"),
        "Views/refit":    diag.get("mean_views_per_refit"),
        "Refits":         diag.get("n_refits"),
        "Note":           "[mock]" if diag.get("mock") else "",
    })

# ML bar reference
rows.append({
    "Strategy": "── ML bar (session 3 best) ──",
    "Sharpe": 2.579, "Ann Ret": None, "Ann Vol": None,
    "Max DD": None, "Turnover/refit": None, "Views/refit": None,
    "Refits": None, "Note": "reference",
})

# Top strategies from existing ML comparison table for context
ml_csv = ROOT / "data/cache/portfolio_returns/ml_strategies_comparison_table.csv"
if ml_csv.exists():
    ml_df = pd.read_csv(ml_csv)
    for _, r in ml_df.head(3).iterrows():
        rows.append({
            "Strategy": r["Strategy"],
            "Sharpe": r["Sharpe"],
            "Ann Ret": r.get("Ann Ret"),
            "Ann Vol": r.get("Ann Vol"),
            "Max DD":  r.get("Max DD"),
            "Turnover/refit": None, "Views/refit": None,
            "Refits": None, "Note": "prior ML/classical",
        })

comp = (
    pd.DataFrame(rows)
    .sort_values("Sharpe", ascending=False, na_position="last")
    .reset_index(drop=True)
)

fmt = {
    "Sharpe":         "{:.3f}",
    "Ann Ret":        "{:.3f}",
    "Ann Vol":        "{:.3f}",
    "Max DD":         "{:.3f}",
    "Turnover/refit": "{:.4f}",
    "Views/refit":    "{:.1f}",
}
comp.style.format(fmt, na_rep="—").set_caption("BL view-generator comparison (OOS 2023-01-01 →)")

## 3 — Equity curves

In [ ]:
if not returns_dict:
    print("No results to plot. Run the backtest scripts first.")
else:
    fig, ax = plt.subplots(figsize=(13, 5))

    for gen, ret in returns_dict.items():
        cum = (1 + ret.dropna()).cumprod()
        label = DISPLAY[gen]
        if diag_dict.get(gen, {}).get("mock"):
            label += " [mock]"
        ax.plot(cum.index, cum.values, label=label, color=COLORS[gen], linewidth=1.6)

    ax.axhline(1.0, color="#888", linewidth=0.8, linestyle="--")
    ax.set_title("BL view-generator variants — OOS equity curves", fontsize=12)
    ax.set_xlabel("Date")
    ax.set_ylabel("Growth of $1")
    ax.legend(loc="upper left", fontsize=9)
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.grid(True, alpha=0.25)
    fig.tight_layout()
    plt.show()

## 4 — Drawdown comparison

In [ ]:
if not returns_dict:
    print("No results to plot.")
else:
    fig, ax = plt.subplots(figsize=(13, 3))

    for gen, ret in returns_dict.items():
        cum = (1 + ret.dropna()).cumprod()
        dd = (cum - cum.cummax()) / cum.cummax()
        ax.fill_between(dd.index, dd.values, 0, alpha=0.25, color=COLORS[gen])
        ax.plot(dd.index, dd.values, color=COLORS[gen], linewidth=0.8, label=DISPLAY[gen])

    ax.set_title("Drawdown", fontsize=12)
    ax.set_xlabel("Date")
    ax.set_ylabel("Drawdown")
    ax.legend(loc="lower left", fontsize=8)
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.grid(True, alpha=0.25)
    fig.tight_layout()
    plt.show()

## 5 — LLM view diagnostics

In [ ]:
llm_gens = [g for g in ["llm-anthropic", "llm-openai"] if g in diag_dict]

if not llm_gens:
    print("No LLM results yet.\n"
          "Run:\n"
          "  python scripts/run_bl_views_backtest.py --generator llm-anthropic\n"
          "  python scripts/run_bl_views_backtest.py --generator llm-openai")
else:
    # Summary table
    llm_rows = []
    for gen in llm_gens:
        d = diag_dict[gen]
        llm_rows.append({
            "Generator":      DISPLAY[gen],
            "Refits":         d.get("n_refits"),
            "Views/refit":    d.get("mean_views_per_refit"),
            "Turnover/refit": d.get("mean_turnover_per_refit"),
            "Sharpe":         d.get("sharpe_ratio"),
            "Mock":           d.get("mock"),
        })
    display(pd.DataFrame(llm_rows))

    # Load per-refit weights to show mean allocation by asset over time
    fig, axes = plt.subplots(1, len(llm_gens), figsize=(7 * len(llm_gens), 4))
    if len(llm_gens) == 1:
        axes = [axes]

    for ax, gen in zip(axes, llm_gens):
        wpath = RESULTS_DIR / f"{gen}_weights.parquet"
        if not wpath.exists():
            ax.text(0.5, 0.5, "weights missing", ha="center", transform=ax.transAxes)
            continue
        wdf = pd.read_parquet(wpath)
        # Top 8 assets by mean allocation
        top8 = wdf.mean().nlargest(8).index.tolist()
        wdf[top8].plot.bar(ax=ax, stacked=False, width=0.7, legend=True)
        ax.set_title(f"{DISPLAY[gen]}\nmean weight per refit (top 8 assets)", fontsize=9)
        ax.set_xlabel("Refit date")
        ax.set_ylabel("Weight")
        ax.tick_params(axis="x", rotation=45, labelsize=7)
        ax.legend(fontsize=7, loc="upper right")

    fig.tight_layout()
    plt.show()

## 6 — Turnover and refit count summary

In [ ]:
if not diag_dict:
    print("No diagnostics available yet.")
else:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3))

    gens_avail = [g for g in GENERATORS if g in diag_dict]
    labels = [DISPLAY[g] for g in gens_avail]
    sharpes = [diag_dict[g].get("sharpe_ratio") or 0.0 for g in gens_avail]
    turnovers = [diag_dict[g].get("mean_turnover_per_refit") or 0.0 for g in gens_avail]
    bar_colors = [COLORS[g] for g in gens_avail]

    ax1.barh(labels, sharpes, color=bar_colors)
    ax1.axvline(2.579, color="black", linestyle="--", linewidth=1, label="ML bar")
    ax1.set_title("OOS Sharpe ratio")
    ax1.legend(fontsize=8)
    ax1.grid(True, axis="x", alpha=0.3)

    ax2.barh(labels, turnovers, color=bar_colors)
    ax2.set_title("Mean turnover per refit")
    ax2.grid(True, axis="x", alpha=0.3)

    fig.tight_layout()
    plt.show()

## Findings

> **Template — fill in after the live backtest.**

### Summary

| Generator | Sharpe | Ann Ret | Ann Vol | Max DD |
|---|---|---|---|---|
| BL-Equilibrium | [TBD] | [TBD] | [TBD] | [TBD] |
| BL-Momentum | [TBD] | [TBD] | [TBD] | [TBD] |
| BL-MeanRev | [TBD] | [TBD] | [TBD] | [TBD] |
| BL-LLM(Anthropic) | [TBD] | [TBD] | [TBD] | [TBD] |
| BL-LLM(OpenAI) | [TBD] | [TBD] | [TBD] | [TBD] |
| ML bar (reference) | 2.579 | — | — | — |

### Key questions to address
1. Do LLM views add Sharpe versus the rule-based BL variants?
2. Does BL-LLM clear or approach the 2.579 ML bar?
3. Which LLM provider produces more consistent/higher-quality views?
4. Is turnover reasonable relative to transaction cost budget (~10 bps)?

### Interpretation

[Fill after live run]

### Next steps

[Fill after live run]